# Manuscript Figure Generation

This notebook generates all figures for the manuscript.

## Input
- Summary CSVs from Results Analysis: `results/summaries/{task}_{model}_ft_{finetune}_summary.csv`
- Embedding regression results: `results/embeddings/`

## Output
- Publication-ready figures in `figures/`

## 1. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# Install adjustText if needed
try:
    from adjustText import adjust_text
except ImportError:
    !pip install adjustText
    from adjustText import adjust_text

# Configuration
RESULTS_DIR = '../results/summaries'
FIGURES_DIR = '../figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# Tasks and models
TASKS = ['Bandgap', 'Dielectric', 'CrystalStructure', 'MatKG']
FINETUNE_CONDITIONS = ['base', 'Bandgap', 'Dielectric', 'CrystalStructure', 'MatKG']

# Plot settings
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 12
sns.set(style='whitegrid', font_scale=1.2)

## 2. Analysis Helper Functions

In [ ]:
def analyze_regression_task(df):
    """
    Analyze regression task results (Bandgap, Dielectric).
    
    Returns:
        maes: List of MAE per iteration
        rmses: List of RMSE per iteration
        mode_mae: MAE using mode prediction
        mode_rmse: RMSE using mode prediction
        entropy_mean: Mean prediction entropy
        entropy_std: Std of prediction entropy
    """
    n_preds = len([col for col in df.columns if 'prediction_' in col 
                   and col != 'mode_prediction' and col != 'prediction_entropy'])
    maes = []
    rmses = []
    for i in range(1, n_preds + 1):
        pred_col = f'prediction_{i}'
        if pred_col in df.columns:
            valid = df[[pred_col, 'Ground']].dropna()
            if len(valid) > 0:
                maes.append(np.abs(valid[pred_col] - valid['Ground']).mean())
                rmses.append(np.sqrt(((valid[pred_col] - valid['Ground'])**2).mean()))
    
    # Mode predictions
    valid = df[['mode_prediction', 'Ground']].dropna()
    mode_mae = np.abs(valid['mode_prediction'] - valid['Ground']).mean() if len(valid) > 0 else np.nan
    mode_rmse = np.sqrt(((valid['mode_prediction'] - valid['Ground'])**2).mean()) if len(valid) > 0 else np.nan
    
    # Entropy
    entropy_mean = df['prediction_entropy'].mean() if 'prediction_entropy' in df.columns else np.nan
    entropy_std = df['prediction_entropy'].std() if 'prediction_entropy' in df.columns else np.nan
    
    return maes, rmses, mode_mae, mode_rmse, entropy_mean, entropy_std


def analyze_classification_task(df):
    """
    Analyze classification task results (CrystalStructure).
    
    Returns:
        accs: List of accuracy per iteration
        mode_acc: Accuracy using mode prediction
        entropy_mean: Mean prediction entropy
        entropy_std: Std of prediction entropy
    """
    n_preds = len([col for col in df.columns if 'prediction_' in col 
                   and col != 'mode_prediction' and col != 'prediction_entropy'])
    accs = []
    for i in range(1, n_preds + 1):
        pred_col = f'prediction_{i}'
        if pred_col in df.columns:
            valid = df[[pred_col, 'Ground']].dropna()
            if len(valid) > 0:
                accs.append((valid[pred_col] == valid['Ground']).mean())
    
    # Mode prediction
    valid = df[['mode_prediction', 'Ground']].dropna()
    mode_acc = (valid['mode_prediction'] == valid['Ground']).mean() if len(valid) > 0 else np.nan
    
    # Entropy
    entropy_mean = df['prediction_entropy'].mean() if 'prediction_entropy' in df.columns else np.nan
    entropy_std = df['prediction_entropy'].std() if 'prediction_entropy' in df.columns else np.nan
    
    return accs, mode_acc, entropy_mean, entropy_std

In [ ]:
def get_model_family(model):
    """
    Categorize model by family for visualization.
    """
    model_lower = model.lower()
    if 'llama-3' in model_lower:
        return 'Llama-3'
    elif 'llama-2' in model_lower or 'llama' in model_lower:
        return 'Llama-2'
    elif 'gpt-4' in model_lower or 'gpt-3' in model_lower:
        return 'GPT'
    elif 'mistral' in model_lower:
        return 'Mistral'
    elif 'mixtral' in model_lower:
        return 'Mixtral'
    elif 'zephyr' in model_lower:
        return 'Zephyr'
    elif 'phi' in model_lower:
        return 'Phi'
    elif 'gemma' in model_lower:
        return 'Gemma'
    elif 'deepseek' in model_lower or 'qwen' in model_lower:
        return 'DeepSeek/Qwen'
    elif 'codellama' in model_lower:
        return 'CodeLlama'
    else:
        return 'Other'


def get_model_size(model):
    """
    Extract model size in billions of parameters.
    """
    match = re.search(r'(\d+)(b)', model.lower())
    if match:
        return int(match.group(1))
    elif 'mini' in model.lower():
        return 2
    elif '8x7b' in model.lower():
        return 56  # Mixtral effective size
    else:
        return 8  # default

## 3. Load All Summary Data

In [ ]:
def load_all_summaries(results_dir):
    """
    Load all summary files and compute metrics.
    
    Returns:
        DataFrame with columns: task, model, finetuned, mode_mae, mode_rmse, 
                                mode_acc, entropy_mean, top1_mean, top5_mean, etc.
    """
    records = []
    
    for filename in os.listdir(results_dir):
        if not filename.endswith('_summary.csv'):
            continue
        
        try:
            # Parse filename: {task}_{model}_ft_{finetune}_summary.csv
            parts = filename.replace('_summary.csv', '').split('_ft_')
            if len(parts) != 2:
                continue
            
            task_model = parts[0]
            ft_condition = parts[1]
            
            # Extract task and model
            task = None
            model = None
            for t in TASKS:
                if task_model.startswith(t + '_'):
                    task = t
                    model = task_model[len(t) + 1:]
                    break
            
            if task is None:
                continue
            
            # Load data
            filepath = os.path.join(results_dir, filename)
            df = pd.read_csv(filepath)
            
            rec = {
                'task': task,
                'model': model,
                'finetuned': ft_condition,
                'path': filepath
            }
            
            # Task-specific metrics
            if task == 'MatKG':
                # MatKG has entropy at different ranks and top-k metrics
                for i in range(1, 6):
                    col = f'entropy_rank_{i}'
                    rec[f'entropy_rank_{i}_mean'] = df[col].mean() if col in df.columns else np.nan
                rec['top1_mean'] = df['top1_rank1_is_1'].mean() if 'top1_rank1_is_1' in df.columns else np.nan
                rec['top5_mean'] = df['avg_top5'].mean() if 'avg_top5' in df.columns else np.nan
                rec['mode_acc'] = np.nan
                rec['entropy_mean'] = np.nan
                rec['mode_mae'] = np.nan
                rec['mode_rmse'] = np.nan
                
            elif task in ['Bandgap', 'Dielectric']:
                # Regression tasks
                _, _, mode_mae, mode_rmse, ent_mean, _ = analyze_regression_task(df)
                rec['mode_mae'] = mode_mae
                rec['mode_rmse'] = mode_rmse
                rec['entropy_mean'] = ent_mean
                rec['mode_acc'] = np.nan
                for i in range(1, 6):
                    rec[f'entropy_rank_{i}_mean'] = np.nan
                rec['top1_mean'] = np.nan
                rec['top5_mean'] = np.nan
                
            elif task == 'CrystalStructure':
                # Classification task
                _, mode_acc, ent_mean, _ = analyze_classification_task(df)
                rec['mode_acc'] = mode_acc
                rec['entropy_mean'] = ent_mean
                rec['mode_mae'] = np.nan
                rec['mode_rmse'] = np.nan
                for i in range(1, 6):
                    rec[f'entropy_rank_{i}_mean'] = np.nan
                rec['top1_mean'] = np.nan
                rec['top5_mean'] = np.nan
            
            records.append(rec)
            
        except Exception as e:
            print(f"Skipping {filename}: {e}")
    
    return pd.DataFrame(records)

# Load all summaries
summary_df = load_all_summaries(RESULTS_DIR)
print(f"Loaded {len(summary_df)} summary records")
summary_df.head()

In [ ]:
# Add model family and size
summary_df['family'] = summary_df['model'].apply(get_model_family)
summary_df['size'] = summary_df['model'].apply(get_model_size)

# Check coverage
print("Tasks:", summary_df['task'].unique())
print("Models:", summary_df['model'].nunique())
print("Finetune conditions:", summary_df['finetuned'].unique())

## 4. Figure 1: Base vs Fine-tuned Performance

In [ ]:
def plot_base_vs_finetuned(summary_df, task, metric, ylabel, title, filename):
    """
    Bar chart comparing base vs fine-tuned performance.
    """
    subset = summary_df[summary_df['task'] == task]
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Get all models
    models = sorted(subset['model'].unique())
    x = np.arange(len(models))
    width = 0.35
    
    # Base values
    base_df = subset[subset['finetuned'] == 'base'].set_index('model')
    base_vals = [base_df.loc[m, metric] if m in base_df.index else np.nan for m in models]
    
    # Fine-tuned values (same task)
    ft_df = subset[subset['finetuned'] == task].set_index('model')
    ft_vals = [ft_df.loc[m, metric] if m in ft_df.index else np.nan for m in models]
    
    ax.bar(x - width/2, base_vals, width, label='Base', alpha=0.8, color='steelblue')
    ax.bar(x + width/2, ft_vals, width, label=f'Fine-tuned ({task})', alpha=0.8, color='coral')
    
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_title(title, fontsize=16, weight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=45, ha='right', fontsize=10)
    ax.legend(fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Bandgap: MAE comparison
plot_base_vs_finetuned(
    summary_df, 'Bandgap', 'mode_mae',
    'Mean Absolute Error (eV)', 
    'Bandgap Prediction: Base vs Fine-tuned',
    'fig1a_bandgap_base_vs_ft.png'
)

In [ ]:
# Dielectric: MAE comparison
plot_base_vs_finetuned(
    summary_df, 'Dielectric', 'mode_mae',
    'Mean Absolute Error',
    'Dielectric Constant Prediction: Base vs Fine-tuned',
    'fig1b_dielectric_base_vs_ft.png'
)

In [ ]:
# CrystalStructure: Accuracy comparison
plot_base_vs_finetuned(
    summary_df, 'CrystalStructure', 'mode_acc',
    'Accuracy',
    'Crystal Structure Classification: Base vs Fine-tuned',
    'fig1c_crystalstructure_base_vs_ft.png'
)

In [ ]:
# MatKG: Top-1 comparison
plot_base_vs_finetuned(
    summary_df, 'MatKG', 'top1_mean',
    'Top-1 Accuracy',
    'MatKG Link Prediction: Base vs Fine-tuned',
    'fig1d_matkg_base_vs_ft.png'
)

## 5. Figure 2: Entropy vs Performance Scatter Plots

In [ ]:
def plot_entropy_vs_performance(summary_df, task, perf_metric, perf_label, 
                                 entropy_col='entropy_mean', title=None, filename=None):
    """
    Scatter plot of entropy vs performance with model family colors.
    """
    # Filter for base and same-task fine-tuned
    plot_df = summary_df[
        (summary_df['task'] == task) & 
        (summary_df['finetuned'].isin(['base', task]))
    ].copy()
    
    if len(plot_df) == 0:
        print(f"No data for {task}")
        return
    
    marker_dict = {'base': 'o', task: 's'}
    
    unique_families = plot_df['family'].unique()
    palette = dict(zip(unique_families, sns.color_palette('Set2', n_colors=len(unique_families))))
    
    # Compute medians
    base_data = plot_df[plot_df['finetuned'] == 'base']
    ft_data = plot_df[plot_df['finetuned'] == task]
    
    base_med_ent = base_data[entropy_col].median() if len(base_data) > 0 else np.nan
    base_med_perf = base_data[perf_metric].median() if len(base_data) > 0 else np.nan
    ft_med_ent = ft_data[entropy_col].median() if len(ft_data) > 0 else np.nan
    ft_med_perf = ft_data[perf_metric].median() if len(ft_data) > 0 else np.nan
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    family_labels_done = set()
    for ft_type in ['base', task]:
        data = plot_df[plot_df['finetuned'] == ft_type]
        for fam in unique_families:
            fam_data = data[data['family'] == fam]
            if len(fam_data) == 0:
                continue
            label = fam if fam not in family_labels_done else None
            ax.scatter(
                fam_data[entropy_col], fam_data[perf_metric],
                marker=marker_dict[ft_type],
                color=palette[fam],
                edgecolors='k', linewidth=1.2,
                s=fam_data['size'] * 10 + 60, alpha=0.85,
                label=label
            )
            family_labels_done.add(fam)
    
    # Median lines
    if not np.isnan(base_med_perf):
        ax.axhline(base_med_perf, color='#377eb8', linestyle='--', linewidth=1.5, alpha=0.8)
    if not np.isnan(base_med_ent):
        ax.axvline(base_med_ent, color='#377eb8', linestyle=':', linewidth=1.5, alpha=0.8)
    if not np.isnan(ft_med_perf):
        ax.axhline(ft_med_perf, color='#e41a1c', linestyle='--', linewidth=1.5, alpha=0.8)
    if not np.isnan(ft_med_ent):
        ax.axvline(ft_med_ent, color='#e41a1c', linestyle=':', linewidth=1.5, alpha=0.8)
    
    # Annotate some points
    n_annotate = min(7, len(plot_df))
    annotate_rows = plot_df.sample(n=n_annotate, random_state=42)
    texts = []
    for _, row in annotate_rows.iterrows():
        texts.append(
            ax.text(
                row[entropy_col], row[perf_metric], row['model'],
                fontsize=9, color='black',
                bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', boxstyle='round,pad=0.15')
            )
        )
    adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5), ax=ax)
    
    ax.set_xlabel('Prediction Entropy (Mean)', fontsize=14)
    ax.set_ylabel(perf_label, fontsize=14)
    ax.set_title(title or f'{task}: {perf_label} vs Entropy', fontsize=16, weight='bold')
    ax.grid(True, linestyle=':', alpha=0.5)
    
    # Legends
    handles_color = [Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[fam],
                            markeredgecolor='k', markersize=12, label=fam) 
                     for fam in unique_families]
    legend1 = ax.legend(handles=handles_color, title='Model Family', fontsize=10,
                        loc='upper left', bbox_to_anchor=(1, 1))
    
    handles_shape = [
        Line2D([0], [0], marker='o', color='k', linestyle='None', markersize=10, label='Base'),
        Line2D([0], [0], marker='s', color='k', linestyle='None', markersize=10, label=f'Fine-Tuned ({task})')
    ]
    legend2 = ax.legend(handles=handles_shape, title='Condition', fontsize=10,
                        loc='upper left', bbox_to_anchor=(1, 0.6))
    ax.add_artist(legend1)
    
    plt.tight_layout()
    if filename:
        plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Bandgap
plot_entropy_vs_performance(
    summary_df, 'Bandgap', 'mode_mae', 'MAE (eV)',
    title='Bandgap: MAE vs Prediction Entropy',
    filename='fig2a_bandgap_entropy_vs_mae.png'
)

In [ ]:
# Dielectric
plot_entropy_vs_performance(
    summary_df, 'Dielectric', 'mode_mae', 'MAE',
    title='Dielectric: MAE vs Prediction Entropy',
    filename='fig2b_dielectric_entropy_vs_mae.png'
)

In [ ]:
# CrystalStructure
plot_entropy_vs_performance(
    summary_df, 'CrystalStructure', 'mode_acc', 'Accuracy',
    title='Crystal Structure: Accuracy vs Prediction Entropy',
    filename='fig2c_crystalstructure_entropy_vs_acc.png'
)

In [ ]:
# MatKG
plot_entropy_vs_performance(
    summary_df, 'MatKG', 'top1_mean', 'Top-1 Accuracy',
    entropy_col='entropy_rank_1_mean',
    title='MatKG: Top-1 Accuracy vs Entropy',
    filename='fig2d_matkg_entropy_vs_top1.png'
)

## 6. Figure 3: Cross-Task Transfer Heatmaps

In [ ]:
def plot_crosseval_heatmap(summary_df, eval_task, metric, metric_label, 
                           title=None, filename=None, lower_is_better=True):
    """
    Heatmap showing cross-task evaluation (ΔMetric relative to base).
    """
    task_df = summary_df[summary_df['task'] == eval_task]
    
    # Define FT order
    ft_order = ['base', eval_task] + [t for t in TASKS if t != eval_task]
    ft_order = [ft for ft in ft_order if ft in task_df['finetuned'].unique()]
    
    # Pivot
    heatmap_df = task_df.pivot(index='model', columns='finetuned', values=metric)
    heatmap_df = heatmap_df.reindex(columns=[c for c in ft_order if c in heatmap_df.columns])
    
    # Compute delta relative to base
    if 'base' in heatmap_df.columns:
        delta_df = heatmap_df.subtract(heatmap_df['base'], axis=0)
    else:
        delta_df = heatmap_df
    
    # Color limits
    min_val = np.nanmin(delta_df.values)
    max_val = np.nanmax(delta_df.values)
    max_abs = max(abs(min_val), abs(max_val))
    vlim = max_abs
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, max(8, 0.4 * len(delta_df))))
    
    cmap = 'coolwarm' if lower_is_better else 'coolwarm_r'
    sns.heatmap(
        delta_df, annot=True, fmt='.3f', center=0,
        cmap=cmap, vmin=-vlim, vmax=vlim,
        cbar_kws={'label': f'Δ{metric_label} (vs base)'},
        ax=ax
    )
    
    ax.set_ylabel('Model', fontsize=12)
    ax.set_xlabel('Fine-tune Task', fontsize=12)
    ax.set_title(title or f'{eval_task}: Cross-Task Evaluation', fontsize=14, weight='bold')
    
    plt.tight_layout()
    if filename:
        plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Bandgap cross-evaluation
plot_crosseval_heatmap(
    summary_df, 'Bandgap', 'mode_mae', 'MAE',
    title='Bandgap: ΔMAE vs Base (Cross-Task Evaluation)',
    filename='fig3a_bandgap_crosseval.png',
    lower_is_better=True
)

In [ ]:
# Dielectric cross-evaluation
plot_crosseval_heatmap(
    summary_df, 'Dielectric', 'mode_mae', 'MAE',
    title='Dielectric: ΔMAE vs Base (Cross-Task Evaluation)',
    filename='fig3b_dielectric_crosseval.png',
    lower_is_better=True
)

In [ ]:
# CrystalStructure cross-evaluation
plot_crosseval_heatmap(
    summary_df, 'CrystalStructure', 'mode_acc', 'Accuracy',
    title='Crystal Structure: ΔAccuracy vs Base (Cross-Task Evaluation)',
    filename='fig3c_crystalstructure_crosseval.png',
    lower_is_better=False
)

In [ ]:
# MatKG cross-evaluation
plot_crosseval_heatmap(
    summary_df, 'MatKG', 'top1_mean', 'Top-1 Acc',
    title='MatKG: ΔTop-1 Accuracy vs Base (Cross-Task Evaluation)',
    filename='fig3d_matkg_crosseval.png',
    lower_is_better=False
)

## 7. Figure 4: MatKG Entropy Difference Heatmap

In [ ]:
def plot_matkg_entropy_heatmap(summary_df, filename=None):
    """
    Heatmap showing entropy difference (fine-tuned - base) for MatKG.
    """
    matkg_df = summary_df[summary_df['task'] == 'MatKG']
    entropy_cols = [f'entropy_rank_{i}_mean' for i in range(1, 6)]
    
    # Base and fine-tuned
    base = matkg_df[matkg_df['finetuned'] == 'base'].set_index('model')[entropy_cols]
    ft = matkg_df[matkg_df['finetuned'] == 'MatKG'].set_index('model')[entropy_cols]
    
    # Common models
    common = base.index.intersection(ft.index)
    if len(common) == 0:
        print("No common models found")
        return
    
    base = base.loc[common]
    ft = ft.loc[common]
    
    # Delta
    delta = ft - base
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, max(8, 0.4 * len(common))))
    
    max_abs = delta.abs().max().max()
    cax = ax.imshow(
        delta.values, aspect='auto', cmap='coolwarm',
        vmin=-max_abs, vmax=max_abs
    )
    
    ax.set_yticks(range(len(common)))
    ax.set_yticklabels(common, fontsize=10)
    ax.set_xticks(range(5))
    ax.set_xticklabels([f'Rank {i}' for i in range(1, 6)], fontsize=11)
    
    plt.colorbar(cax, label='ΔEntropy (Fine-Tuned − Base)', ax=ax)
    ax.set_title('MatKG: Entropy Difference by Rank', fontsize=14, weight='bold')
    
    plt.tight_layout()
    if filename:
        plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=300, bbox_inches='tight')
    plt.show()

plot_matkg_entropy_heatmap(summary_df, filename='fig4_matkg_entropy_diff.png')

## 8. Figure 5: MatKG Per-Relation Analysis

In [ ]:
def load_matkg_relation_data(summary_df):
    """
    Load per-relation metrics for MatKG.
    """
    def canonical_rel(rel):
        items = rel.split('-')
        return '-'.join(sorted(items))
    
    records = []
    matkg_paths = summary_df[summary_df['task'] == 'MatKG']['path'].dropna()
    
    for path in matkg_paths:
        try:
            fname = os.path.basename(path)
            parts = fname.replace('_summary.csv', '').split('_ft_')
            model = parts[0].replace('MatKG_', '')
            ft = parts[1] if len(parts) > 1 else 'unknown'
            
            df = pd.read_csv(path)
            if 'rel' not in df.columns:
                continue
            
            df['canonical_rel'] = df['rel'].apply(canonical_rel)
            
            agg_cols = {}
            for col in ['entropy_rank_1', 'entropy_rank_2', 'entropy_rank_3', 
                        'entropy_rank_4', 'entropy_rank_5', 'top1_rank1_is_1', 'avg_top5']:
                if col in df.columns:
                    agg_cols[col] = 'mean'
            
            if len(agg_cols) == 0:
                continue
            
            agg = df.groupby('canonical_rel').agg(agg_cols).reset_index()
            agg['model'] = model
            agg['finetuned'] = ft
            records.append(agg)
            
        except Exception as e:
            print(f"Error processing {path}: {e}")
    
    if len(records) == 0:
        return pd.DataFrame()
    
    return pd.concat(records, ignore_index=True)

matkg_rel_df = load_matkg_relation_data(summary_df)
print(f"Loaded {len(matkg_rel_df)} relation records")

In [ ]:
def plot_matkg_relation_heatmap(matkg_rel_df, ft_condition, filename=None):
    """
    Heatmap of Top-1 accuracy per relation for a given fine-tune condition.
    """
    if len(matkg_rel_df) == 0:
        print("No relation data available")
        return
    
    plot_df = matkg_rel_df[matkg_rel_df['finetuned'] == ft_condition].copy()
    
    if 'top1_rank1_is_1' not in plot_df.columns:
        print(f"No top1 data for {ft_condition}")
        return
    
    # Pivot: relations (rows) x models (columns)
    heatmap = plot_df.pivot(index='canonical_rel', columns='model', values='top1_rank1_is_1')
    
    # Sort models by mean accuracy
    model_means = heatmap.mean(axis=0).sort_values(ascending=False)
    heatmap = heatmap[model_means.index]
    
    # Sort relations by mean accuracy
    rel_means = heatmap.mean(axis=1).sort_values(ascending=False)
    heatmap = heatmap.loc[rel_means.index]
    
    # Plot
    fig, ax = plt.subplots(figsize=(min(18, 2 + 0.8 * heatmap.shape[1]), 12))
    
    sns.heatmap(
        heatmap, annot=False, cmap='Blues',
        vmin=0, vmax=1, cbar_kws={'label': 'Top-1 Accuracy'},
        ax=ax
    )
    
    ax.set_ylabel('Relation', fontsize=12)
    ax.set_xlabel('Model', fontsize=12)
    ax.set_title(f'MatKG Top-1 Accuracy per Relation ({ft_condition})', fontsize=14, weight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(fontsize=9)
    
    plt.tight_layout()
    if filename:
        plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=300, bbox_inches='tight')
    plt.show()

# Base models
plot_matkg_relation_heatmap(matkg_rel_df, 'base', filename='fig5a_matkg_base_relations.png')

In [ ]:
# Fine-tuned models
plot_matkg_relation_heatmap(matkg_rel_df, 'MatKG', filename='fig5b_matkg_ft_relations.png')

## 9. Figure 6: Model Ranking Summary

In [ ]:
def create_ranking_table(summary_df):
    """
    Create a summary table ranking models across all tasks.
    """
    rankings = []
    
    for task in TASKS:
        task_df = summary_df[
            (summary_df['task'] == task) & 
            (summary_df['finetuned'] == task)
        ]
        
        if task in ['Bandgap', 'Dielectric']:
            metric = 'mode_mae'
            ascending = True  # Lower is better
        elif task == 'CrystalStructure':
            metric = 'mode_acc'
            ascending = False  # Higher is better
        else:  # MatKG
            metric = 'top1_mean'
            ascending = False
        
        ranked = task_df.sort_values(metric, ascending=ascending)[['model', metric]].reset_index(drop=True)
        ranked['rank'] = range(1, len(ranked) + 1)
        ranked['task'] = task
        ranked = ranked.rename(columns={metric: 'score'})
        rankings.append(ranked)
    
    return pd.concat(rankings, ignore_index=True)

ranking_df = create_ranking_table(summary_df)
ranking_df.head(20)

In [ ]:
def plot_model_ranking_heatmap(ranking_df, filename=None):
    """
    Heatmap showing model ranks across tasks.
    """
    # Pivot: models x tasks
    rank_pivot = ranking_df.pivot(index='model', columns='task', values='rank')
    
    # Compute average rank
    rank_pivot['Avg Rank'] = rank_pivot.mean(axis=1)
    rank_pivot = rank_pivot.sort_values('Avg Rank')
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, max(8, 0.4 * len(rank_pivot))))
    
    # Exclude Avg Rank for heatmap colors
    plot_data = rank_pivot[TASKS]
    
    sns.heatmap(
        plot_data, annot=True, fmt='.0f', cmap='RdYlGn_r',
        cbar_kws={'label': 'Rank (1 = best)'},
        ax=ax
    )
    
    ax.set_ylabel('Model', fontsize=12)
    ax.set_xlabel('Task', fontsize=12)
    ax.set_title('Model Rankings Across Tasks (Fine-tuned)', fontsize=14, weight='bold')
    
    plt.tight_layout()
    if filename:
        plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=300, bbox_inches='tight')
    plt.show()
    
    return rank_pivot

rank_summary = plot_model_ranking_heatmap(ranking_df, filename='fig6_model_rankings.png')
rank_summary.head(10)

## 10. Figure 7: Improvement Analysis

In [ ]:
def compute_improvement_stats(summary_df):
    """
    Compute improvement from base to fine-tuned for each task.
    """
    improvements = []
    
    for task in TASKS:
        task_df = summary_df[summary_df['task'] == task]
        
        if task in ['Bandgap', 'Dielectric']:
            metric = 'mode_mae'
            better = 'lower'
        elif task == 'CrystalStructure':
            metric = 'mode_acc'
            better = 'higher'
        else:
            metric = 'top1_mean'
            better = 'higher'
        
        for model in task_df['model'].unique():
            model_df = task_df[task_df['model'] == model]
            
            base_row = model_df[model_df['finetuned'] == 'base']
            ft_row = model_df[model_df['finetuned'] == task]
            
            if len(base_row) > 0 and len(ft_row) > 0:
                base_val = base_row[metric].values[0]
                ft_val = ft_row[metric].values[0]
                
                if better == 'lower':
                    improvement = (base_val - ft_val) / base_val * 100 if base_val != 0 else np.nan
                else:
                    improvement = (ft_val - base_val) / base_val * 100 if base_val != 0 else np.nan
                
                improvements.append({
                    'task': task,
                    'model': model,
                    'base': base_val,
                    'finetuned': ft_val,
                    'improvement_pct': improvement
                })
    
    return pd.DataFrame(improvements)

improvement_df = compute_improvement_stats(summary_df)
improvement_df.head()

In [ ]:
def plot_improvement_distribution(improvement_df, filename=None):
    """
    Box plot showing improvement distribution per task.
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    sns.boxplot(
        data=improvement_df, x='task', y='improvement_pct',
        palette='Set2', ax=ax
    )
    
    ax.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.set_xlabel('Task', fontsize=12)
    ax.set_ylabel('Improvement (%)', fontsize=12)
    ax.set_title('Performance Improvement: Base → Fine-tuned', fontsize=14, weight='bold')
    
    plt.tight_layout()
    if filename:
        plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=300, bbox_inches='tight')
    plt.show()

plot_improvement_distribution(improvement_df, filename='fig7_improvement_boxplot.png')

In [ ]:
# Summary statistics
print("\nImprovement Summary by Task:")
print(improvement_df.groupby('task')['improvement_pct'].agg(['mean', 'std', 'min', 'max']).round(2))

## 11. Export Summary Tables

In [ ]:
# Export ranking table
rank_summary.to_csv(os.path.join(FIGURES_DIR, 'table_model_rankings.csv'))

# Export improvement table
improvement_df.to_csv(os.path.join(FIGURES_DIR, 'table_improvements.csv'), index=False)

# Export full summary
summary_df.to_csv(os.path.join(FIGURES_DIR, 'table_full_summary.csv'), index=False)

print(f"Tables exported to {FIGURES_DIR}")

## Notes

### Figure Descriptions

1. **Figure 1 (a-d)**: Base vs Fine-tuned comparison bar charts for each task
2. **Figure 2 (a-d)**: Entropy vs Performance scatter plots showing the relationship between prediction consistency and accuracy
3. **Figure 3 (a-d)**: Cross-task evaluation heatmaps showing transfer learning effects
4. **Figure 4**: MatKG entropy difference heatmap across prediction ranks
5. **Figure 5 (a-b)**: MatKG per-relation accuracy heatmaps (base vs fine-tuned)
6. **Figure 6**: Model ranking summary across all tasks
7. **Figure 7**: Improvement distribution from base to fine-tuned

### Interpretation Guide

- **Lower entropy** = More consistent predictions across iterations
- **Positive improvement %** = Fine-tuning helped
- **Cross-task transfer**: Negative ΔMAE means improvement; positive ΔAccuracy means improvement